# 05 — Virtual Screening

Score the FDA-approved compound library with the trained classifier.
Rank hits by p(active) and inspect their 2-D structures.

**Input:** trained model + ChEMBL FDA-approved library
**Output:** `../data/processed/screening_hits.csv`, `top_hits_structures.png`

In [ ]:
import sys, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '..')
from src.screening import fetch_fda_approved_library, screen_library
from src.features import compute_fingerprints, lipinski_filter
from src.models import load_model

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

## Download FDA-approved compound library

Pulls `max_phase = 4` small molecules from ChEMBL (~2,500 compounds).
The CSV is cached after the first run.

In [ ]:
library_path = '../data/processed/fda_library.csv'

if Path(library_path).exists():
    library_df = pd.read_csv(library_path)
    print(f"Loaded cached library: {len(library_df)} compounds")
else:
    library_df = fetch_fda_approved_library(library_path)
    print(f"Downloaded and cached: {len(library_df)} compounds")

library_df.head()

## Screen against the trained model

Pipeline: Lipinski filter → ECFP4 fingerprints → predict p(active) → rank.

In [ ]:
model_path = '../data/processed/models/combined_model.joblib'

hits = screen_library(
    library_df=library_df,
    model_path=model_path,
    apply_lipinski=True,
    top_n=config['screening']['top_n_hits'],
)

hits.to_csv('../data/processed/screening_hits.csv', index=False)
print(f"Top {len(hits)} hits saved to data/processed/screening_hits.csv")
hits[['rank', 'pref_name', 'molecule_chembl_id', 'p_active']].head(20)

## Score distribution across the full library

In [ ]:
model = load_model(model_path)
lib_filt = lipinski_filter(library_df)
X_lib, _ = compute_fingerprints(lib_filt)
all_probs = model.predict_proba(X_lib)[:, 1]

cutoff = hits['p_active'].min()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(all_probs, bins=60, color='steelblue', alpha=0.75)
ax.axvline(cutoff, color='red', ls='--', label=f'Top-{len(hits)} cutoff ({cutoff:.3f})')
ax.set_xlabel('p(active)')
ax.set_ylabel('Compound count')
ax.set_title('Predicted activity scores: FDA-approved library')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/screening_score_dist.png', dpi=150)
plt.show()

## Visualise top hit structures

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

top12 = hits.head(12)
mols   = [Chem.MolFromSmiles(s) for s in top12['canonical_smiles']]
labels = [
    f"{row['pref_name'] or row['molecule_chembl_id']}\np={row['p_active']:.3f}"
    for _, row in top12.iterrows()
]

img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(320, 260), legends=labels)
img.save('../data/processed/top_hits_structures.png')
display(img)

## Summary

| Notebook | Output |
|---|---|
| 01 Target Acquisition | `target_manifest.csv` |
| 02 Ligand Retrieval | `combined_bioactivity.csv` |
| 03 Feature Engineering | `features_X.npy`, UMAP plot |
| 04 Classifier Training | `models/combined_model.joblib`, ROC/PR curves |
| 05 Virtual Screening | `screening_hits.csv`, `top_hits_structures.png` |

**Next step →** share the top 10–20 hits with your HGSOC collaborators.
Prioritise compounds already approved for other indications (drug repurposing).